In [1]:
tabla='qtiop10'
col_tabla = 'infopefec'

In [2]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2

In [3]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [6]:
# fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
# fecha= fecha.iloc[0, 0]
# print(fecha)

now = datetime.now()

# query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"

# c1= text(query)
# connection2.execute(c1)

In [5]:
fecha = '01/05/23'
#fecha= '2023-05-01'

In [7]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()


query0 = f"""
select
d1.INFOPEORICENASICOD,
d1.INFOPECENASICOD,
d1.INFOPESOLOPENUM,
d1.INFOPESECNUM,
--INFOPEFEC,
to_char(d1.INFOPEFEC, 'YYYY-MM-DD HH24:MI:SS') as INFOPEFEC,
--d1.INFOPEINGSOPHOR,
to_char(d1.INFOPEINGSOPHOR, 'YYYY-MM-DD HH24:MI:SS') as INFOPEINGSOPHOR,
--d1.INFOPESOPTPO,
to_char(d1.INFOPESOPTPO, 'YYYY-MM-DD HH24:MI:SS') as INFOPESOPTPO,
--d1.INFOPEANETPO,
to_char(d1.INFOPEANETPO, 'YYYY-MM-DD HH24:MI:SS') as INFOPEANETPO,
--d1.INFOPEOPETPO,
to_char(d1.INFOPEOPETPO, 'YYYY-MM-DD HH24:MI:SS') as INFOPEOPETPO,
d1.INFOPEEXAPATFLG,
d1.TIPHOPCOD,
d1.INFOPEUSUCRECOD,
--d1.INFOPECREFEC,
to_char(d1.INFOPECREFEC, 'YYYY-MM-DD HH24:MI:SS') as INFOPECREFEC,
INFOPEUSUMODCOD,
--d1.INFOPEMODFEC,
to_char(d1.INFOPEMODFEC, 'YYYY-MM-DD HH24:MI:SS') as INFOPEMODFEC,
d1.DESESOCOD,
d1.INFOPEACTMEDOPENUM,
d1.INFOPEBTB,
d1.INFOPENROGES,
d1.INFOPENROPART,
d1.INFOPECESAANTE,
d1.INFOPETRABPART,
d1.INFOPEEXPUL,
d1.INFOPEDARES,

a1.solopefec as solopefec,  
a1.SOLOPEACTMEDNUM,
a1.SOLOPEBUSPACSECNUM

from {tabla} d1 
left outer join qtsop10 a1 on a1.SOLOPEORICENASICOD = d1.INFOPEORICENASICOD
and a1.SOLOPECENASICOD    = d1.INFOPECENASICOD
and a1.SOLOPENUM    = d1.INFOPESOLOPENUM
where d1.{col_tabla}>='{fecha}'

"""

base2 = pd.read_sql_query(query0,con=connection0)


connection0.close()

In [8]:
base2.shape

(48513, 27)

In [9]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48513 entries, 0 to 48512
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   infopeoricenasicod  48513 non-null  object        
 1   infopecenasicod     48513 non-null  object        
 2   infopesolopenum     48513 non-null  int64         
 3   infopesecnum        48513 non-null  int64         
 4   infopefec           48513 non-null  object        
 5   infopeingsophor     48513 non-null  object        
 6   infopesoptpo        48513 non-null  object        
 7   infopeanetpo        48513 non-null  object        
 8   infopeopetpo        48513 non-null  object        
 9   infopeexapatflg     48513 non-null  object        
 10  tiphopcod           48513 non-null  object        
 11  infopeusucrecod     48513 non-null  object        
 12  infopecrefec        48513 non-null  object        
 13  infopeusumodcod     22265 non-null  object    

In [10]:
#CREAMOS LA TABLA TEMPORAL
base2.to_sql(name=f'tmp_{tabla}', con=engine3, if_exists='replace', index=False)

513

In [11]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48513 entries, 0 to 48512
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   infopeoricenasicod  48513 non-null  object        
 1   infopecenasicod     48513 non-null  object        
 2   infopesolopenum     48513 non-null  int64         
 3   infopesecnum        48513 non-null  int64         
 4   infopefec           48513 non-null  object        
 5   infopeingsophor     48513 non-null  object        
 6   infopesoptpo        48513 non-null  object        
 7   infopeanetpo        48513 non-null  object        
 8   infopeopetpo        48513 non-null  object        
 9   infopeexapatflg     48513 non-null  object        
 10  tiphopcod           48513 non-null  object        
 11  infopeusucrecod     48513 non-null  object        
 12  infopecrefec        48513 non-null  object        
 13  infopeusumodcod     22265 non-null  object    

In [12]:
#LA SUBIMOS AL POSTGRES SQL

query_count_before = text(f"SELECT COUNT(*) FROM {tabla}")
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")


Cantidad de filas en la tabla qtiop10 antes de la inserción: 936726


In [13]:
#Borramos en caso el ETL se ejecute una segunda vez
borrando=text(f"DELETE FROM {tabla} WHERE {col_tabla} >='{fecha}'")
borrado = connection3.execute(borrando)

In [22]:
query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN infopeoricenasicod TYPE character(1),
ALTER COLUMN infopecenasicod TYPE character(3),
ALTER COLUMN infopesolopenum TYPE numeric(10,0),
ALTER COLUMN infopesecnum TYPE numeric(2,0),
ALTER COLUMN infopefec TYPE date USING infopefec::date,
ALTER COLUMN infopeingsophor TYPE timestamp USING infopeingsophor::timestamp without time zone,
ALTER COLUMN infopesoptpo TYPE timestamp USING infopesoptpo::timestamp without time zone,
ALTER COLUMN infopeanetpo TYPE timestamp USING infopeanetpo::timestamp without time zone,
ALTER COLUMN infopeopetpo TYPE timestamp USING infopeopetpo::timestamp without time zone,
ALTER COLUMN infopeexapatflg TYPE character(1),
ALTER COLUMN tiphopcod TYPE character(1),
ALTER COLUMN infopeusucrecod TYPE character(10),
ALTER COLUMN infopecrefec TYPE timestamp USING infopecrefec::timestamp without time zone,
ALTER COLUMN infopeusumodcod TYPE character(10),
ALTER COLUMN infopemodfec TYPE timestamp USING infopemodfec::timestamp without time zone,
ALTER COLUMN desesocod TYPE character(2),
ALTER COLUMN infopeactmedopenum TYPE numeric(10,0),
ALTER COLUMN infopebtb TYPE numeric(1,0),
ALTER COLUMN infopenroges TYPE numeric(2,0),
ALTER COLUMN infopenropart TYPE numeric(2,0),
ALTER COLUMN infopecesaante TYPE character(2),
ALTER COLUMN infopetrabpart TYPE numeric(1,0),
ALTER COLUMN infopeexpul TYPE numeric(1,0),
ALTER COLUMN infopedares TYPE numeric(1,0),
ALTER COLUMN solopefec TYPE date,
ALTER COLUMN solopeactmednum TYPE numeric(10,0),
ALTER COLUMN solopebuspacsecnum TYPE numeric(10,0);


INSERT INTO {tabla} 
(infopeoricenasicod,infopecenasicod,infopesolopenum,infopesecnum,infopefec,infopeingsophor,infopesoptpo,infopeanetpo,infopeopetpo,infopeexapatflg,tiphopcod,infopeusucrecod,infopecrefec,infopeusumodcod,infopemodfec,desesocod,infopeactmedopenum,infopebtb,infopenroges,infopenropart,infopecesaante,infopetrabpart,infopeexpul,infopedares,solopefec,solopeactmednum,solopebuspacsecnum) 

SELECT 
infopeoricenasicod,infopecenasicod,infopesolopenum,infopesecnum,infopefec,infopeingsophor,infopesoptpo,infopeanetpo,infopeopetpo,infopeexapatflg,tiphopcod,infopeusucrecod,infopecrefec,infopeusumodcod,infopemodfec,desesocod,infopeactmedopenum,infopebtb,infopenroges,infopenropart,infopecesaante,infopetrabpart,infopeexpul,infopedares,solopefec,solopeactmednum,solopebuspacsecnum

FROM tmp_{tabla};
"""

c1= text(query)
connection3.execute(c1)

In [23]:
query_count_after = text(f"SELECT COUNT(*) FROM {tabla}")
cant_despues = connection3.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla qtiop10 después de la inserción: 937941
La cantidad de filas insertadas fue de: 1215


In [24]:
#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
"""
c2= text(query2)
cursor=connection3.execute(c2)

In [25]:
connection1.close()
connection2.close()
connection3.close()

In [27]:
engine1.dispose()
engine2.dispose()
engine3.dispose()

AYUDA PARA EXTRAER COLUMNAS Y ESTRUCTURA DE TABLAS (NO ES PARTE DEL ETL)

In [1]:
import re

cadena = """
    infopeoricenasicod character(1) COLLATE pg_catalog."default",
    infopecenasicod character(3) COLLATE pg_catalog."default",
    infopesolopenum numeric(10,0),
    infopesecnum numeric(2,0),
    infopefec date,
    infopeingsophor timestamp with time zone,
    infopesoptpo timestamp with time zone,
    infopeanetpo timestamp with time zone,
    infopeopetpo timestamp with time zone,
    infopeexapatflg character(1) COLLATE pg_catalog."default",
    tiphopcod character(1) COLLATE pg_catalog."default",
    infopeusucrecod character(10) COLLATE pg_catalog."default",
    infopecrefec timestamp with time zone,
    infopeusumodcod character(10) COLLATE pg_catalog."default",
    infopemodfec timestamp with time zone,
    desesocod character(2) COLLATE pg_catalog."default",
    infopeactmedopenum numeric(10,0),
    infopebtb numeric(1,0),
    infopenroges numeric(2,0),
    infopenropart numeric(2,0),
    infopecesaante character(2) COLLATE pg_catalog."default",
    infopetrabpart numeric(1,0),
    infopeexpul numeric(1,0),
    infopedares numeric(1,0),
    solopefec date,
    solopeactmednum numeric(10,0),
    solopebuspacsecnum numeric(10,0)
"""

# Reemplaza los caracteres innecesarios
cadena = cadena.replace(" COLLATE pg_catalog.\"default\",", "")
cadena = cadena.replace(" with time zone", "")

# Divide la cadena en una lista de líneas
lineas = cadena.split("\n")

# Crea la cadena de alteración de columnas
cadena_alter = ""
for i, linea in enumerate(lineas):
    palabras = linea.split()
    if len(palabras) >= 2:
        columna = palabras[0]
        tipo = palabras[1]
        if i == len(lineas) - 2:
            # Última línea, agrega punto y coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo};\n"
        else:
            # Otras líneas, agrega coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo},\n"

# Utiliza una expresión regular para eliminar las comas duplicadas
cadena_alter = re.sub(r',+$', ',', cadena_alter, flags=re.MULTILINE)

print(cadena_alter)



import re

nombrecitos = re.findall(r'ALTER COLUMN\s+(\S+)', cadena_alter)
insertado_col = ",".join(nombrecitos)

print(insertado_col)

ALTER COLUMN infopeoricenasicod TYPE character(1),
ALTER COLUMN infopecenasicod TYPE character(3),
ALTER COLUMN infopesolopenum TYPE numeric(10,0),
ALTER COLUMN infopesecnum TYPE numeric(2,0),
ALTER COLUMN infopefec TYPE date,
ALTER COLUMN infopeingsophor TYPE timestamp,
ALTER COLUMN infopesoptpo TYPE timestamp,
ALTER COLUMN infopeanetpo TYPE timestamp,
ALTER COLUMN infopeopetpo TYPE timestamp,
ALTER COLUMN infopeexapatflg TYPE character(1),
ALTER COLUMN tiphopcod TYPE character(1),
ALTER COLUMN infopeusucrecod TYPE character(10),
ALTER COLUMN infopecrefec TYPE timestamp,
ALTER COLUMN infopeusumodcod TYPE character(10),
ALTER COLUMN infopemodfec TYPE timestamp,
ALTER COLUMN desesocod TYPE character(2),
ALTER COLUMN infopeactmedopenum TYPE numeric(10,0),
ALTER COLUMN infopebtb TYPE numeric(1,0),
ALTER COLUMN infopenroges TYPE numeric(2,0),
ALTER COLUMN infopenropart TYPE numeric(2,0),
ALTER COLUMN infopecesaante TYPE character(2),
ALTER COLUMN infopetrabpart TYPE numeric(1,0),
ALTER COL